# DB Development General

---

## 1. RDBMS

A **Relational Database Management System (RDBMS)** is software that stores and manages structured data organized into tables with defined relationships. It enforces data integrity, supports SQL, and handles concurrent access safely.

**Popular RDBMS:** PostgreSQL, MySQL, SQL Server, Oracle, SQLite.

---

## 2. Tables, Columns, Rows

| Concept | Description |
|--------|-------------|
| **Table** | A named collection of related data, like a spreadsheet |
| **Column** | A named field with a specific data type (e.g., `name VARCHAR`, `age INT`) |
| **Row** | A single record in the table |

```sql
-- Table: employees
| emp_id | name    | salary  |
|--------|---------|---------|
|   1    | Alice   | 5000.00 |
|   2    | Bob     | 4500.00 |
```

---

## 3. Relations — Primary Key & Foreign Key

- **Primary Key (PK):** Uniquely identifies each row. Cannot be NULL or duplicated.
- **Foreign Key (FK):** References a PK in another table, establishing a relationship and enforcing referential integrity.

```sql
CREATE TABLE departments (
  dept_id INT PRIMARY KEY,
  name    VARCHAR(100)
);

CREATE TABLE employees (
  emp_id  INT PRIMARY KEY,
  name    VARCHAR(100),
  dept_id INT REFERENCES departments(dept_id)  -- FK
);
```

---

## 4. Views

A **view** is a saved SQL query stored as a virtual table. It does not store data itself — it re-runs the query on access. Useful for simplifying complex queries, reusability, and access control.

```sql
CREATE VIEW high_earners AS
  SELECT name, salary FROM employees WHERE salary > 8000;

SELECT * FROM high_earners;
```

> **Materialized views** (in some RDBMS) physically cache the result and need to be refreshed — useful for expensive queries.

---

## 5. Data Manipulation — Novice

### INSERT — Add new rows
```sql
INSERT INTO employees (emp_id, name, salary)
VALUES (3, 'Carol', 6000.00);
```

### UPDATE — Modify existing rows
```sql
UPDATE employees
SET salary = 6500.00
WHERE emp_id = 3;
```

### DELETE — Remove rows
```sql
DELETE FROM employees
WHERE emp_id = 3;
```

> ⚠️ Always use `WHERE` with `UPDATE`/`DELETE` to avoid affecting the entire table.

---

## 6. Data Selection — Novice

### Simple Selections
```sql
SELECT name, salary
FROM employees
WHERE salary > 5000
ORDER BY salary DESC
LIMIT 10;
```

### Joins
Joins combine rows from two or more tables based on a related column.

| Type | Returns |
|------|---------|
| `INNER JOIN` | Only rows with matches in **both** tables |
| `LEFT JOIN` | All rows from the left + matching from right (NULL if no match) |
| `RIGHT JOIN` | All rows from the right + matching from left |
| `FULL OUTER JOIN` | All rows from both, NULL where no match |

```sql
-- INNER JOIN example
SELECT e.name, d.name AS department
FROM employees e
INNER JOIN departments d ON e.dept_id = d.dept_id;

-- LEFT JOIN: includes employees without a department
SELECT e.name, d.name AS department
FROM employees e
LEFT JOIN departments d ON e.dept_id = d.dept_id;
```

---

## 7. Data Definition Language (DDL)

DDL commands define and modify the database **structure** (schema), not the data.

### CREATE — Define a new object
```sql
CREATE TABLE projects (
  project_id INT          PRIMARY KEY,
  title      VARCHAR(200) NOT NULL,
  budget     DECIMAL(10,2)
);
```

### DROP — Permanently remove an object
```sql
DROP TABLE projects;       -- removes table and all its data
DROP VIEW high_earners;
```

### ALTER — Modify an existing structure
```sql
ALTER TABLE employees ADD COLUMN hire_date DATE;
ALTER TABLE employees DROP COLUMN hire_date;
ALTER TABLE employees ALTER COLUMN salary TYPE NUMERIC(12,2);
```

---

## 8. Transactions — ACID

A **transaction** is a sequence of operations treated as a single unit of work. Either all succeed (`COMMIT`) or all are undone (`ROLLBACK`).

```sql
BEGIN;
  UPDATE accounts SET balance = balance - 100 WHERE id = 1;
  UPDATE accounts SET balance = balance + 100 WHERE id = 2;
COMMIT;
-- On error: ROLLBACK;
```

### ACID Properties

**Atomicity** — All or nothing. If any step fails, the entire transaction is rolled back.
> Transfer $100: debit and credit both happen, or neither does.

**Consistency** — The DB moves from one valid state to another, always respecting constraints and rules.
> A balance can't go negative if a `CHECK (balance >= 0)` constraint exists.

**Isolation** — Concurrent transactions don't see each other's intermediate states. Controlled via isolation levels.

**Durability** — Once committed, data survives crashes (written to disk / WAL log).

---

## 9. Transaction Isolation Levels

Isolation levels control what a transaction can **see from other concurrent transactions**. Higher isolation = fewer anomalies, but more locking overhead.

### Anomalies Explained
| Anomaly | Description |
|---------|-------------|
| **Dirty Read** | Reading uncommitted data from another transaction |
| **Non-Repeatable Read** | Same row returns different values within the same transaction |
| **Phantom Read** | A re-executed query returns new rows added by another transaction |

### Levels vs. Anomalies

| Level | Dirty Read | Non-Repeatable Read | Phantom Read |
|-------|:----------:|:-------------------:|:------------:|
| `READ UNCOMMITTED` | ✅ Possible | ✅ Possible | ✅ Possible |
| `READ COMMITTED` | ❌ Prevented | ✅ Possible | ✅ Possible |
| `REPEATABLE READ` | ❌ Prevented | ❌ Prevented | ✅ Possible |
| `SERIALIZABLE` | ❌ Prevented | ❌ Prevented | ❌ Prevented |

```sql
SET TRANSACTION ISOLATION LEVEL REPEATABLE READ;
BEGIN;
  SELECT salary FROM employees WHERE emp_id = 1;  -- returns 5000
  -- another session updates salary to 6000 and commits
  SELECT salary FROM employees WHERE emp_id = 1;  -- still returns 5000 (protected)
COMMIT;
```

---

## 10. CAP Theorem

In a **distributed database**, you can only guarantee **two of three** properties simultaneously:

| Property | Meaning |
|----------|---------|
| **Consistency (C)** | Every read returns the most recent write |
| **Availability (A)** | Every request gets a response (no timeout/error) |
| **Partition Tolerance (P)** | System works despite network failures/splits |

Since network partitions are unavoidable in distributed systems, the real choice is between **CP** (consistent, may reject requests) or **AP** (always responds, data may be stale).

| Choice | Examples | Behavior during partition |
|--------|----------|---------------------------|
| **CP** | HBase, Zookeeper, MongoDB | Refuses requests to stay consistent |
| **AP** | Cassandra, CouchDB, DynamoDB | Responds with possibly stale data |

---

## 11. Cursors

A **cursor** allows iterating over a query result **row by row**, useful when procedural row-level logic is required that set-based SQL can't handle easily.

```sql
DECLARE emp_cursor CURSOR FOR
  SELECT emp_id, salary FROM employees;

OPEN emp_cursor;
FETCH NEXT FROM emp_cursor INTO @id, @salary;

WHILE @@FETCH_STATUS = 0
BEGIN
  -- process each row individually
  FETCH NEXT FROM emp_cursor INTO @id, @salary;
END;

CLOSE emp_cursor;
DEALLOCATE emp_cursor;
```

> ⚠️ Cursors are much slower than set-based operations on large datasets. Prefer set-based SQL whenever possible.

---

## 12. Stored Procedures & Functions

Both are named, reusable blocks of SQL logic stored in the database.

| Aspect | Stored Procedure | Function |
|--------|-----------------|----------|
| Returns | Optional / output params | Always returns a value |
| Side effects | Yes (INSERT, UPDATE, etc.) | Ideally none |
| Used in SELECT | No (most RDBMS) | Yes |
| Called with | `CALL` / `EXEC` | Inside a query expression |

```sql
-- Stored Procedure
CREATE PROCEDURE give_raise(IN p_emp_id INT, IN p_amount DECIMAL)
BEGIN
  UPDATE employees SET salary = salary + p_amount WHERE emp_id = p_emp_id;
END;

CALL give_raise(1, 500);

-- Function
CREATE FUNCTION annual_salary(monthly DECIMAL)
RETURNS DECIMAL
BEGIN
  RETURN monthly * 12;
END;

SELECT name, annual_salary(salary) AS annual FROM employees;
```

---

## 13. Indexes

An **index** is a data structure (usually a B-tree) built on one or more columns to speed up data retrieval by avoiding full table scans.

```sql
-- Single-column index
CREATE INDEX idx_emp_salary ON employees(salary);

-- Composite index (useful for queries filtering on both columns)
CREATE INDEX idx_dept_salary ON employees(dept_id, salary);

-- Unique index (enforces uniqueness + speeds up lookups)
CREATE UNIQUE INDEX idx_emp_email ON employees(email);
```

| Type | Description |
|------|-------------|
| **Clustered** | Defines physical row order on disk (one per table) |
| **Non-Clustered** | Separate structure pointing to rows (many per table) |
| **Composite** | Index on multiple columns; order matters |
| **Covering** | Includes all columns a query needs (no table lookup) |

> **Trade-off:** Indexes speed up `SELECT` but slow down `INSERT`, `UPDATE`, and `DELETE` because the index must be maintained on every write.

---

## 14. Triggers

A **trigger** is a stored routine that automatically fires in response to a table event (`INSERT`, `UPDATE`, `DELETE`), either `BEFORE` or `AFTER` the event.

```sql
CREATE TRIGGER trg_salary_audit
AFTER UPDATE ON employees
FOR EACH ROW
BEGIN
  IF OLD.salary <> NEW.salary THEN
    INSERT INTO salary_audit (emp_id, old_salary, new_salary, changed_at)
    VALUES (OLD.emp_id, OLD.salary, NEW.salary, NOW());
  END IF;
END;
```

**Common use cases:** audit logging, enforcing complex business rules, cascading updates, automatic timestamp management.

---

## 15. Data Manipulation — Intermediate

### MERGE / UPSERT

Inserts a row if it doesn't exist, or updates it if it does — in a single atomic statement.

```sql
-- PostgreSQL: ON CONFLICT
INSERT INTO employees (emp_id, name, salary)
VALUES (1, 'Alice', 5500)
ON CONFLICT (emp_id)
DO UPDATE SET salary = EXCLUDED.salary;

-- SQL Server / Standard SQL: MERGE
MERGE INTO employees AS target
USING (VALUES (1, 'Alice', 5500)) AS source(emp_id, name, salary)
  ON target.emp_id = source.emp_id
WHEN MATCHED THEN
  UPDATE SET salary = source.salary
WHEN NOT MATCHED THEN
  INSERT (emp_id, name, salary)
  VALUES (source.emp_id, source.name, source.salary);
```

---

## 16. Data Selection — Intermediate

### UNION / UNION ALL
Combines result sets from two queries that have compatible column structures.

```sql
-- UNION: removes duplicate rows (slower)
SELECT name FROM employees
UNION
SELECT name FROM contractors;

-- UNION ALL: keeps duplicates, faster
SELECT name FROM employees
UNION ALL
SELECT name FROM contractors;
```

### Aggregations

**GROUP BY** — Groups rows and applies aggregate functions per group.
```sql
SELECT dept_id,
       COUNT(*)     AS headcount,
       AVG(salary)  AS avg_salary,
       MAX(salary)  AS top_salary
FROM employees
GROUP BY dept_id
HAVING AVG(salary) > 5000;  -- filter on aggregate (not WHERE)
```

**DISTINCT** — Returns only unique values.
```sql
SELECT DISTINCT dept_id FROM employees;
```

**Window Functions** — Perform calculations across related rows **without collapsing** them (unlike `GROUP BY`). Defined with `OVER()`.

```sql
SELECT
  name,
  dept_id,
  salary,
  RANK()      OVER (PARTITION BY dept_id ORDER BY salary DESC) AS dept_rank,
  AVG(salary) OVER (PARTITION BY dept_id)                      AS dept_avg,
  SUM(salary) OVER (ORDER BY emp_id ROWS UNBOUNDED PRECEDING)  AS running_total
FROM employees;
```

Key window functions: `RANK()`, `DENSE_RANK()`, `ROW_NUMBER()`, `LAG()`, `LEAD()`, `FIRST_VALUE()`, `SUM()`, `AVG()`.

### Subqueries
A query nested inside another query. Can appear in `SELECT`, `FROM`, or `WHERE`.

```sql
-- Scalar subquery in WHERE
SELECT name, salary FROM employees
WHERE salary > (SELECT AVG(salary) FROM employees);

-- IN subquery
SELECT name FROM employees
WHERE dept_id IN (
  SELECT dept_id FROM departments WHERE location = 'NYC'
);

-- Correlated subquery (references outer query row by row)
SELECT name, salary FROM employees e
WHERE salary = (
  SELECT MAX(salary) FROM employees WHERE dept_id = e.dept_id
);
```

---

## 17. Data Architecture — Intermediate

### Database Normalization Forms

Normalization organizes tables to **reduce redundancy** and **improve data integrity** through progressive rules called Normal Forms.

| Form | Rule |
|------|------|
| **1NF** | All values are atomic (no arrays/lists); each row is unique |
| **2NF** | 1NF + no partial dependency on a composite primary key |
| **3NF** | 2NF + no transitive dependency (non-key column depending on another non-key column) |
| **BCNF** | Every determinant is a candidate key (stricter than 3NF) |

```
-- ❌ Violates 1NF (non-atomic column):
| emp_id | skills              |
|--------|---------------------|
|   1    | SQL, Python, Java   |

-- ✅ Fixed to 1NF (atomic rows):
| emp_id | skill  |
|--------|--------|
|   1    | SQL    |
|   1    | Python |
|   1    | Java   |
```

---

### Explain Plans

`EXPLAIN` (or `EXPLAIN ANALYZE`) reveals the **execution plan** the query optimizer chose — essential for diagnosing slow queries.

```sql
EXPLAIN ANALYZE
SELECT e.name, d.name
FROM employees e
JOIN departments d ON e.dept_id = d.dept_id
WHERE e.salary > 5000;
```

**Key nodes to look for:**

| Node | What it Means |
|------|---------------|
| `Seq Scan` | Full table scan — consider adding an index |
| `Index Scan` | Using an index — efficient |
| `Hash Join` / `Nested Loop` | Join strategy chosen by the optimizer |
| `cost=X..Y` | Estimated startup and total cost |
| `rows=N` | Estimated rows returned |
| `actual time` | Real execution time (only with `ANALYZE`) |

---

### Partitioning

**Partitioning** splits a large table into smaller physical segments (partitions) while presenting as one logical table. Queries targeting a specific range skip irrelevant partitions (**partition pruning**).

| Strategy | Split By | Best For |
|----------|----------|----------|
| **Range** | Value ranges | Dates, numeric ranges |
| **List** | Discrete values | Country, status, category |
| **Hash** | Hash of column | Even distribution, no natural range |

```sql
-- PostgreSQL: Range partitioning by year
CREATE TABLE orders (
  order_id   INT,
  amount     DECIMAL,
  order_date DATE
) PARTITION BY RANGE (order_date);

CREATE TABLE orders_2024 PARTITION OF orders
  FOR VALUES FROM ('2024-01-01') TO ('2025-01-01');

CREATE TABLE orders_2025 PARTITION OF orders
  FOR VALUES FROM ('2025-01-01') TO ('2026-01-01');
```

**Benefits:** faster scans on range queries, easier archiving (drop a whole partition), improved maintenance and vacuuming.

---

## 18. Data Architecture — Advanced

### Database Denormalization

**Denormalization** intentionally reintroduces redundancy into a schema to improve **read performance** at the cost of storage and write complexity. It's the deliberate reversal of normalization rules, optimized for query speed.

**When to use:**
- Read-heavy workloads (reporting, dashboards, analytics)
- JOINs across many normalized tables become a bottleneck
- OLAP / Data Warehouse systems (vs. OLTP which prefers normalized)

```sql
-- Normalized: requires 2 JOINs per query
SELECT o.order_id, c.name AS customer, p.title AS product, o.amount
FROM orders o
JOIN customers c ON o.customer_id = c.id
JOIN products  p ON o.product_id  = p.id;

-- Denormalized: flat table with redundant columns — no JOIN needed
SELECT order_id, customer_name, product_title, amount
FROM orders_flat;
-- customer_name and product_title are stored directly in orders_flat
```

### Common Denormalization Techniques

| Technique | Description |
|-----------|-------------|
| **Storing aggregates** | Pre-compute and store `total_orders` on the customer row |
| **Duplicating columns** | Copy `customer_name` into `orders` to avoid a JOIN |
| **Flattened summary tables** | Wide tables with pre-aggregated metrics for reporting |
| **Materialized views** | Cached query results refreshed on a schedule or on demand |

### Trade-offs at a Glance

| Aspect | Normalized | Denormalized |
|--------|:----------:|:------------:|
| Storage | Less | More |
| Write performance | Faster | Slower (update many copies) |
| Read performance | Slower (JOINs) | Faster (flat reads) |
| Data integrity | High | Risk of anomalies |
| Best for | **OLTP** | **OLAP / Reporting** |

